In [1]:
import numpy as np
from ler.image_properties import ImageProperties
from gwsnr import GWSNR
from ler.gw_source_population import CBCSourceParameterDistribution


In [2]:
def get_lensed_gw_params(lens_params, num_required_images=2, cosmology=None):
    """
    Computes lensed GW image properties from an unlensed GW and lens galaxy parameters.

    Parameters
    ----------
    lens_params : dict
        Dictionary containing lens galaxy parameters:
            'zl', 'zs', 'theta_E', 'q', 'phi', 'gamma', 'gamma1', 'gamma2', etc.
    num_required_images : int
        Minimum number of lensed images required for acceptance.
    cosmology : astropy.cosmology instance or None
        Cosmology object for distance calculations.

    Returns
    -------
    lensed_gw_params : dict
        Dictionary containing lensed image parameters:
            'x_image_positions', 'y_image_positions', 'magnifications',
            'time_delays', 'image_type', 'n_images', and source parameters
            ('mass_1', 'mass_2', etc.) repeated per image.
        Returns None if lensing fails or fewer than `num_required_images` are found.
    """
    ip = ImageProperties(
        n_min_images=num_required_images,
        n_max_images=4,
        lens_model_list=['EPL_NUMBA', 'SHEAR'],
        cosmology=cosmology
    )
  
    try:
        lensed_output = ip.image_properties(lens_params)
    except Exception as e:
        print("Lens equation solver failed:", e)
        return None

    if lensed_output['n_images'][0] < num_required_images:
        return None

    return lensed_output

In [3]:
lens_parameters = dict(
    zs=np.array([2.0]),
    zl=np.array([0.5]),
    gamma1=np.array([0.0]),
    gamma2=np.array([0.0]),
    phi=np.array([0.0]),
    q=np.array([0.8]),
    gamma=np.array([2.0]),
    theta_E=np.array([1.0])
)
res = get_lensed_gw_params(lens_parameters, num_required_images=4)

solving lens equations...
n_min_images > 2 is not supported yet
Lens equation solver failed: 


In [4]:
res

In [5]:
cbc = CBCSourceParameterDistribution(
        event_type="BBH",
        spin_zero=False,
        spin_precession=True,
    )

gw_params = cbc.sample_gw_parameters(size=1)


Initializing CBCSourceRedshiftDistribution class...

luminosity_distance interpolator will be loaded from ./interpolator_json/luminosity_distance/luminosity_distance_1.json
differential_comoving_volume interpolator will be loaded from ./interpolator_json/differential_comoving_volume/differential_comoving_volume_1.json
using ler available merger rate density model: merger_rate_density_madau_dickinson_belczynski_ng
merger_rate_density_madau_dickinson_belczynski_ng interpolator will be loaded from ./interpolator_json/merger_rate_density/merger_rate_density_madau_dickinson_belczynski_ng_2.json
merger_rate_density_detector_frame interpolator will be loaded from ./interpolator_json/merger_rate_density/merger_rate_density_detector_frame_3.json
source_redshift interpolator will be loaded from ./interpolator_json/source_redshift/source_redshift_1.json

Initializing CBCSourceParameterDistribution class...

using ler available zs function : source_redshift
using ler available source_frame_masses

In [6]:
gw_params

{'zs': array([1.98039021]),
 'geocent_time': array([1.25567296e+09]),
 'ra': array([0.89033658]),
 'dec': array([-0.82745474]),
 'phase': array([3.30016612]),
 'psi': array([1.91706891]),
 'theta_jn': array([1.24574955]),
 'a_1': array([0.47255731]),
 'a_2': array([0.10672517]),
 'tilt_1': array([2.00388475]),
 'tilt_2': array([2.34269865]),
 'phi_12': array([4.01262675]),
 'phi_jl': array([5.4347853]),
 'luminosity_distance': array([15353.25004453]),
 'mass_1_source': array([7.70452547]),
 'mass_2_source': array([7.09247501]),
 'mass_1': array([22.96249231]),
 'mass_2': array([21.1383431])}

In [7]:
ip = ImageProperties()
a = ip.produce_effective_params({**gw_params, **res})

TypeError: 'NoneType' object is not a mapping

In [ ]:
a

{'zs': array([2.]),
 'geocent_time': array([1.24085896e+09]),
 'ra': array([0.7336363]),
 'dec': array([0.19493385]),
 'phase': array([4.66558655]),
 'psi': array([0.54137233]),
 'theta_jn': array([2.56760181]),
 'a_1': array([0.631903]),
 'a_2': array([0.79778387]),
 'tilt_1': array([2.08806966]),
 'tilt_2': array([1.70378184]),
 'phi_12': array([4.0590643]),
 'phi_jl': array([5.13548593]),
 'luminosity_distance': array([22883.64509195]),
 'mass_1_source': array([7.59324612]),
 'mass_2_source': array([7.53630901]),
 'mass_1': array([28.46913813]),
 'mass_2': array([28.25566547]),
 'zl': array([0.5]),
 'gamma1': array([0.]),
 'gamma2': array([0.]),
 'phi': array([0.]),
 'q': array([0.8]),
 'gamma': array([2.]),
 'theta_E': array([1.]),
 'x0_image_positions': array([[ 0.63558415, -0.05164792,         nan,         nan]]),
 'x1_image_positions': array([[ 1.81894596, -0.10803149,         nan,         nan]]),
 'magnifications': array([[ 1.89959944, -0.14852171,         nan,         nan]]),


In [ ]:
def get_lensed_gw_snrs(
    lensed_gw_params,
    snr_threshold=7.0,
    num_detected_gws=2
):
    snr_calc = GWSNR(waveform_approximant='IMRPhenomXPHM')

    n_images = int(lensed_gw_params['n_images'][0])

    snr_net = []
    snr_H1 = []
    snr_L1 = []
    snr_V1 = []

    # ---- Compute SNR for real images only ----
    for i in range(n_images):

        snr_out = snr_calc.optimal_snr(
            mass_1=lensed_gw_params['mass_1'][0],
            mass_2=lensed_gw_params['mass_2'][0],
            luminosity_distance=lensed_gw_params['effective_luminosity_distance'][0, i],
            theta_jn=lensed_gw_params['theta_jn'][0],
            psi=lensed_gw_params['psi'][0],
            phase=lensed_gw_params['effective_phase'][0, i],
            geocent_time=lensed_gw_params['effective_geocent_time'][0, i],
            ra=lensed_gw_params['effective_ra'][0, i],
            dec=lensed_gw_params['effective_dec'][0, i],
            a_1=lensed_gw_params['a_1'][0],
            a_2=lensed_gw_params['a_2'][0],
            tilt_1=lensed_gw_params['tilt_1'][0],
            tilt_2=lensed_gw_params['tilt_2'][0],
            phi_12=lensed_gw_params['phi_12'][0],
            phi_jl=lensed_gw_params['phi_jl'][0]
        )

        snr_net.append(snr_out['optimal_snr_net'][0])
        snr_H1.append(snr_out['optimal_snr_H1'][0])
        snr_L1.append(snr_out['optimal_snr_L1'][0])
        snr_V1.append(snr_out['optimal_snr_V1'][0])

    snr_net = np.array(snr_net)
    snr_H1  = np.array(snr_H1)
    snr_L1  = np.array(snr_L1)
    snr_V1  = np.array(snr_V1)

    # Detection logic 
    detected_mask = snr_net >= snr_threshold
    n_detected = detected_mask.sum()

    # Reject if too few
    if n_detected < num_detected_gws:
        return False, {
            'optimal_snr_net': snr_net,
            'optimal_snr_H1': snr_H1,
            'optimal_snr_L1': snr_L1,
            'optimal_snr_V1': snr_V1
        }

    # If too many, keep highest SNR ones
    if n_detected > num_detected_gws:
        sorted_idx = np.argsort(snr_net)[::-1]
        keep_idx = sorted_idx[:num_detected_gws]
    else:
        keep_idx = np.where(detected_mask)[0]

    # ---- Filter per-image arrays (shape 1×4) ----
    image_keys = [
        'x0_image_positions',
        'x1_image_positions',
        'magnifications',
        'time_delays',
        'image_type',
        'effective_luminosity_distance',
        'effective_geocent_time',
        'effective_phase',
        'effective_ra',
        'effective_dec'
    ]

    for key in image_keys:
        arr = lensed_gw_params[key]
        lensed_gw_params[key] = arr[:, keep_idx]

    # Update number of images
    lensed_gw_params['n_images'][0] = len(keep_idx)

    snr_dict = {
        'optimal_snr_net': snr_net[keep_idx],
        'optimal_snr_H1': snr_H1[keep_idx],
        'optimal_snr_L1': snr_L1[keep_idx],
        'optimal_snr_V1': snr_V1[keep_idx]
    }

    return True, snr_dict

In [ ]:
get_lensed_gw_snrs(a)


Initializing GWSNR class...

psds not given. Choosing bilby's default psds
Intel processor has trouble allocating memory when the data is huge. So, by default for IMRPhenomXPHM, duration_max = 64.0. Otherwise, set to some max value like duration_max = 600.0 (10 mins)
Interpolator will be generated for L1 detector at ./interpolator_json/L1/partialSNR_dict_1.json
Interpolator will be generated for H1 detector at ./interpolator_json/H1/partialSNR_dict_1.json
Interpolator will be generated for V1 detector at ./interpolator_json/V1/partialSNR_dict_1.json
Please be patient while the interpolator is generated
Generating interpolator for ['L1', 'H1', 'V1'] detectors


100%|██████████████████████████████████████████████████████| 400000/400000 [07:03<00:00, 943.62it/s]



Saving Partial-SNR for L1 detector with shape (20, 200, 10, 10)

Saving Partial-SNR for H1 detector with shape (20, 200, 10, 10)

Saving Partial-SNR for V1 detector with shape (20, 200, 10, 10)

Chosen GWSNR initialization parameters:

npool:  4
snr type:  interpolation_aligned_spins
waveform approximant:  IMRPhenomXPHM
sampling frequency:  2048.0
minimum frequency (fmin):  20.0
reference frequency (f_ref):  20.0
mtot=mass1+mass2
min(mtot):  9.96
max(mtot) (with the given fmin=20.0): 235.0
detectors:  ['L1', 'H1', 'V1']
psds:  [[array([  10.21659,   10.23975,   10.26296, ..., 4972.81   ,
       4984.081  , 4995.378  ], shape=(2736,)), array([4.43925574e-41, 4.22777986e-41, 4.02102594e-41, ...,
       6.51153524e-46, 6.43165104e-46, 6.55252996e-46],
      shape=(2736,)), <scipy.interpolate._interpolate.interp1d object at 0x7e8ba562ede0>], [array([  10.21659,   10.23975,   10.26296, ..., 4972.81   ,
       4984.081  , 4995.378  ], shape=(2736,)), array([4.43925574e-41, 4.22777986e-41, 4

(False,
 {'optimal_snr_net': array([0.86864052, 0.32916364]),
  'optimal_snr_H1': array([0.55317582, 0.19543053]),
  'optimal_snr_L1': array([0.44278049, 0.24076079]),
  'optimal_snr_V1': array([0.50247219, 0.11040766])})

In [ ]:
get_lensed_gw_snrs(a)


Initializing GWSNR class...

psds not given. Choosing bilby's default psds
Intel processor has trouble allocating memory when the data is huge. So, by default for IMRPhenomXPHM, duration_max = 64.0. Otherwise, set to some max value like duration_max = 600.0 (10 mins)
Interpolator will be loaded for L1 detector from ./interpolator_json/L1/partialSNR_dict_1.json
Interpolator will be loaded for H1 detector from ./interpolator_json/H1/partialSNR_dict_1.json
Interpolator will be loaded for V1 detector from ./interpolator_json/V1/partialSNR_dict_1.json

Chosen GWSNR initialization parameters:

npool:  4
snr type:  interpolation_aligned_spins
waveform approximant:  IMRPhenomXPHM
sampling frequency:  2048.0
minimum frequency (fmin):  20.0
reference frequency (f_ref):  20.0
mtot=mass1+mass2
min(mtot):  9.96
max(mtot) (with the given fmin=20.0): 235.0
detectors:  ['L1', 'H1', 'V1']
psds:  [[array([  10.21659,   10.23975,   10.26296, ..., 4972.81   ,
       4984.081  , 4995.378  ], shape=(2736,)

(False,
 {'optimal_snr_net': array([0.86864052, 0.32916364]),
  'optimal_snr_H1': array([0.55317582, 0.19543053]),
  'optimal_snr_L1': array([0.44278049, 0.24076079]),
  'optimal_snr_V1': array([0.50247219, 0.11040766])})

In [ ]:
def simulate_lensed_gw_detection(
    gw_params,
    lens_params,
    num_required_images=2,
    num_detected_gws=2,
    snr_threshold=7.0,
    cosmology=None,
    waveform_approximant="IMRPhenomXPHM"
):
    """
    Full pipeline:
    1) Solve lens equation
    2) Produce effective lensed GW parameters
    3) Compute SNRs
    4) Apply detection logic

    Returns
    -------
    (False, snr_dict) if detection fails
    (True, snr_dict, lensed_gw_params) if detection passes
    """

    
    # Solve lens equation
    
    ip = ImageProperties(
        n_min_images=num_required_images,
        n_max_images=4,
        lens_model_list=['EPL_NUMBA', 'SHEAR'],
        cosmology=cosmology
    )

    try:
        lensed_output = ip.image_properties(lens_params)
    except Exception as e:
        print("Lens equation solver failed:", e)
        return False, None

    if lensed_output['n_images'][0] < num_required_images:
        return False, None

    # Produce effective GW parameters
    try:
        lensed_gw_params = ip.produce_effective_params(
            {**gw_params, **lensed_output}
        )
    except Exception as e:
        print("Effective parameter computation failed:", e)
        return False, None

    n_images = int(lensed_gw_params['n_images'][0])

    # Compute SNRs

    snr_calc = GWSNR(waveform_approximant=waveform_approximant)

    snr_net = []
    snr_H1 = []
    snr_L1 = []
    snr_V1 = []

    for i in range(n_images):

        snr_out = snr_calc.optimal_snr(
            mass_1=lensed_gw_params['mass_1'][0],
            mass_2=lensed_gw_params['mass_2'][0],
            luminosity_distance=lensed_gw_params['effective_luminosity_distance'][0, i],
            theta_jn=lensed_gw_params['theta_jn'][0],
            psi=lensed_gw_params['psi'][0],
            phase=lensed_gw_params['effective_phase'][0, i],
            geocent_time=lensed_gw_params['effective_geocent_time'][0, i],
            ra=lensed_gw_params['effective_ra'][0, i],
            dec=lensed_gw_params['effective_dec'][0, i],
            a_1=lensed_gw_params['a_1'][0],
            a_2=lensed_gw_params['a_2'][0],
            tilt_1=lensed_gw_params['tilt_1'][0],
            tilt_2=lensed_gw_params['tilt_2'][0],
            phi_12=lensed_gw_params['phi_12'][0],
            phi_jl=lensed_gw_params['phi_jl'][0]
        )

        snr_net.append(snr_out['optimal_snr_net'][0])
        snr_H1.append(snr_out['optimal_snr_H1'][0])
        snr_L1.append(snr_out['optimal_snr_L1'][0])
        snr_V1.append(snr_out['optimal_snr_V1'][0])

    snr_net = np.array(snr_net)
    snr_H1  = np.array(snr_H1)
    snr_L1  = np.array(snr_L1)
    snr_V1  = np.array(snr_V1)

    snr_dict = {
        'optimal_snr_net': snr_net,
        'optimal_snr_H1': snr_H1,
        'optimal_snr_L1': snr_L1,
        'optimal_snr_V1': snr_V1
    }

    # Detection logic
    detected_mask = snr_net >= snr_threshold
    n_detected = detected_mask.sum()

    if n_detected < num_detected_gws:
        return False, snr_dict, lensed_gw_params

    # If more pass than required,keep highest SNR ones
    if n_detected > num_detected_gws:
        sorted_idx = np.argsort(snr_net)[::-1]
        keep_idx = sorted_idx[:num_detected_gws]
    else:
        keep_idx = np.where(detected_mask)[0]


    image_keys = [
        'x0_image_positions',
        'x1_image_positions',
        'magnifications',
        'time_delays',
        'image_type',
        'effective_luminosity_distance',
        'effective_geocent_time',
        'effective_phase',
        'effective_ra',
        'effective_dec'
    ]

    for key in image_keys:
        arr = lensed_gw_params[key]
        lensed_gw_params[key] = arr[:, keep_idx]

    lensed_gw_params['n_images'][0] = len(keep_idx)

    snr_dict = {k: v[keep_idx] for k, v in snr_dict.items()}
    return True, snr_dict, lensed_gw_params

In [ ]:
simulate_lensed_gw_detection(gw_params=gw_params,lens_params=lens_parameters)

solving lens equations...


100%|█████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  8.09it/s]



Initializing GWSNR class...

psds not given. Choosing bilby's default psds
Intel processor has trouble allocating memory when the data is huge. So, by default for IMRPhenomXPHM, duration_max = 64.0. Otherwise, set to some max value like duration_max = 600.0 (10 mins)
Interpolator will be loaded for L1 detector from ./interpolator_json/L1/partialSNR_dict_1.json
Interpolator will be loaded for H1 detector from ./interpolator_json/H1/partialSNR_dict_1.json
Interpolator will be loaded for V1 detector from ./interpolator_json/V1/partialSNR_dict_1.json

Chosen GWSNR initialization parameters:

npool:  4
snr type:  interpolation_aligned_spins
waveform approximant:  IMRPhenomXPHM
sampling frequency:  2048.0
minimum frequency (fmin):  20.0
reference frequency (f_ref):  20.0
mtot=mass1+mass2
min(mtot):  9.96
max(mtot) (with the given fmin=20.0): 235.0
detectors:  ['L1', 'H1', 'V1']
psds:  [[array([  10.21659,   10.23975,   10.26296, ..., 4972.81   ,
       4984.081  , 4995.378  ], shape=(2736,)

(False,
 {'optimal_snr_net': array([2.66399843, 0.4639302 ]),
  'optimal_snr_H1': array([1.83523906, 0.28494535]),
  'optimal_snr_L1': array([1.66240126, 0.29951212]),
  'optimal_snr_V1': array([0.98244963, 0.21054658])},
 {'zs': array([2.]),
  'geocent_time': array([1.24085896e+09]),
  'ra': array([0.7336363]),
  'dec': array([0.19493385]),
  'phase': array([4.66558655]),
  'psi': array([0.54137233]),
  'theta_jn': array([2.56760181]),
  'a_1': array([0.631903]),
  'a_2': array([0.79778387]),
  'tilt_1': array([2.08806966]),
  'tilt_2': array([1.70378184]),
  'phi_12': array([4.0590643]),
  'phi_jl': array([5.13548593]),
  'luminosity_distance': array([22883.64509195]),
  'mass_1_source': array([7.59324612]),
  'mass_2_source': array([7.53630901]),
  'mass_1': array([28.46913813]),
  'mass_2': array([28.25566547]),
  'zl': array([0.5]),
  'gamma1': array([0.]),
  'gamma2': array([0.]),
  'phi': array([0.]),
  'q': array([0.8]),
  'gamma': array([2.]),
  'theta_E': array([1.]),
  'x0_i